# 06 — Pull GDELT Monthly Aggregates (BigQuery)

**Source:** GDELT 2.0 Events (`gdelt-bq.full.events`)  
**Provider:** GDELT Project  
**Access:** Google BigQuery — public dataset, no BQ charges for SELECT on public tables  
**Coverage:** 1979–present (2013+ at 15-minute resolution)  

## What this notebook does
We do **not** pull raw GDELT events (billions of rows). Instead we aggregate directly
inside BigQuery to a **country-month** summary matching Domain 6 of the meta-plan:

| Feature | Description |
|---|---|
| `goldstein_mean` | Monthly mean Goldstein Scale (−10=destabilising, +10=stabilising) |
| `goldstein_std` | Monthly volatility of Goldstein scores |
| `avgtone_mean` | Monthly mean news sentiment |
| `avgtone_std` | Monthly volatility of news sentiment |
| `events_total` | Total event count |
| `events_quad3` | Verbal conflict events (QuadClass=3) |
| `events_quad4` | Material conflict events (QuadClass=4) |
| `events_protests` | CAMEO root code 14 (protest) |
| `events_assaults` | CAMEO root code 18 (assault) |
| `events_fights` | CAMEO root code 19 (fight) |
| `events_mass_violence` | CAMEO root code 20 (mass violence) |
| `mentions_total` | Sum of NumMentions (event salience) |

## Required environment variables
```
ADLS_ACCOUNT_NAME      — Azure storage account name
ADLS_CONTAINER         — Container name (default: 'data')
GCP_PROJECT_ID         — Your GCP project ID (for BigQuery billing)
GOOGLE_APPLICATION_CREDENTIALS  — Path to GCP service account JSON key
                                   (or use Workload Identity in GKE/Cloud Run)
```

## Cost note
The aggregation query scans ~2–5 GB per year. At \$5/TB, a full 2000–2024 pull
costs approximately \$0.50–1.00. Subsequent incremental runs (one year) cost ~\$0.02.

In [ ]:
import os
import pandas as pd
from datetime import datetime
from google.cloud import bigquery
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
# ── Load GCP credentials from Key Vault ────────────────────────────────────
# KEY_VAULT_URL must be set as a compute instance environment variable.
# Writes the GCP service account JSON to a temp file and sets
# GOOGLE_APPLICATION_CREDENTIALS and GCP_PROJECT_ID in os.environ.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent.parent))  # repo root
from utils.keyvault import load_gcp_credentials
_gcp_sa_path = load_gcp_credentials()

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
GCP_PROJECT_ID    = os.environ["GCP_PROJECT_ID"]
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# Pull window — set END_YEAR to current year for incremental updates
START_YEAR = 2000
END_YEAR   = datetime.utcnow().year

print(f"GCP project : {GCP_PROJECT_ID}")
print(f"Pull window : {START_YEAR}–{END_YEAR}")
print(f"Run date    : {RUN_DATE}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## BigQuery aggregation query

We group at country-month granularity using `ActionGeo_CountryCode` (CAMEO country code
of where the action took place — the most reliable geographic field in GDELT).

In [ ]:
GDELT_QUERY = f"""
SELECT
  ActionGeo_CountryCode                       AS cameo_country,
  CAST(SUBSTR(CAST(Day AS STRING), 1, 4) AS INT64) AS year,
  CAST(SUBSTR(CAST(Day AS STRING), 5, 2) AS INT64) AS month,
  FORMAT('%d-%02d', CAST(SUBSTR(CAST(Day AS STRING), 1, 4) AS INT64),
                    CAST(SUBSTR(CAST(Day AS STRING), 5, 2) AS INT64)) AS year_month,

  COUNT(*)                                    AS events_total,
  AVG(GoldsteinScale)                         AS goldstein_mean,
  STDDEV_POP(GoldsteinScale)                  AS goldstein_std,
  AVG(AvgTone)                                AS avgtone_mean,
  STDDEV_POP(AvgTone)                         AS avgtone_std,
  SUM(NumMentions)                            AS mentions_total,

  -- Conflict event sub-counts
  COUNTIF(QuadClass = 3)                      AS events_quad3_verbal_conflict,
  COUNTIF(QuadClass = 4)                      AS events_quad4_material_conflict,
  COUNTIF(EventRootCode = '14')               AS events_protests,
  COUNTIF(EventRootCode = '15')               AS events_force_posture,
  COUNTIF(EventRootCode = '18')               AS events_assaults,
  COUNTIF(EventRootCode = '19')               AS events_fights,
  COUNTIF(EventRootCode = '20')               AS events_mass_violence,

  -- Cooperative event sub-counts (for contrast features)
  COUNTIF(QuadClass = 1)                      AS events_quad1_verbal_coop,
  COUNTIF(QuadClass = 2)                      AS events_quad2_material_coop

FROM `gdelt-bq.full.events`
WHERE
  CAST(SUBSTR(CAST(Day AS STRING), 1, 4) AS INT64) BETWEEN {START_YEAR} AND {END_YEAR}
  AND ActionGeo_CountryCode IS NOT NULL
  AND ActionGeo_CountryCode != ''
GROUP BY 1, 2, 3, 4
ORDER BY cameo_country, year, month
"""

print("Query prepared. Estimated scan: ~2–5 GB/year × ", END_YEAR - START_YEAR + 1, "years")

## Run query via BigQuery

In [ ]:
bq_client = bigquery.Client(project=GCP_PROJECT_ID)

print("Submitting BigQuery job...")
job = bq_client.query(GDELT_QUERY)
print(f"Job ID: {job.job_id}")

# This blocks until the job completes (typically 2–5 minutes for 25 years)
df_gdelt = job.to_dataframe(create_bqstorage_client=True)
print(f"Done. Rows returned: {len(df_gdelt):,}")
df_gdelt.head(3)

## Post-processing

GDELT uses 2-letter CAMEO country codes; we add the ISO3 crosswalk so the GDELT
panel joins cleanly to the master country-month panel.

In [ ]:
# Minimal CAMEO → ISO3 crosswalk for the most common country codes
# Full crosswalk available at: https://www.gdeltproject.org/data/lookups/CAMEO.country.txt
CAMEO_ISO3_URL = "https://www.gdeltproject.org/data/lookups/CAMEO.country.txt"

import requests as _req
import io as _io

try:
    resp_cx = _req.get(CAMEO_ISO3_URL, timeout=30)
    resp_cx.raise_for_status()
    df_cameo = pd.read_csv(
        _io.BytesIO(resp_cx.content), sep="\t", header=None,
        names=["cameo_country", "country_label"],
    )
    # GDELT CAMEO codes approximate ISO2; use pycountry or a manual map for ISO3
    # For now, attach the human-readable label — ISO3 join happens in build_panel.py
    df_gdelt = df_gdelt.merge(df_cameo, on="cameo_country", how="left")
    print(f"Crosswalk applied: {df_cameo['cameo_country'].nunique()} CAMEO codes mapped")
except Exception as e:
    print(f"Could not fetch CAMEO crosswalk ({e}); skipping country label join")

## Write to ADLS

In [ ]:
write_parquet(df_gdelt, f"raw/gdelt/{RUN_DATE}/gdelt_monthly.parquet")

## Summary

In [ ]:
print("=" * 55)
print("GDELT pull complete")
print("=" * 55)
print(f"  Rows            : {len(df_gdelt):,} country-months")
print(f"  CAMEO countries : {df_gdelt['cameo_country'].nunique()}")
print(f"  Year range      : {df_gdelt['year'].min()}–{df_gdelt['year'].max()}")
print(f"  BQ Job ID       : {job.job_id}")
print()
print("ADLS path written:")
print(f"  raw/gdelt/{RUN_DATE}/gdelt_monthly.parquet")